# Final seasonal London parameter sensitivity

This notebook now produces one main sensitivity figure. It changes **one parameter at a time**, keeps every other parameter fixed, and repeats stochastic forecasts at each tested value. This answers: *if this parameter were different within the stated range, how much would the six-week peak change?*

For each parameter the notebook reports two presentation-ready quantities:

1. **Outcome spread:** the largest minus smallest mean forecast across tested values. For outbreak probability this is reported in percentage points.
2. **Relative parameter-variance share:** the parameter's noise-adjusted variance across its level means, divided by the sum of that variance across all parameters. These shares sum to 100% and provide one clear relative ranking.

The variance share is an OAT screening allocation, not a Sobol index or a universal causal percentage. It depends on the tested ranges. Baselines come from the final seasonal Poisson calibration; the default range is ±20%, clipped to declared scientific/scenario bounds. The forecast-origin hidden state is estimated once from the latest four observations and then held fixed across levels, so reconditioning cannot conceal a parameter effect.

The joint-design analysis is also enabled for the final run. It changes parameters together and complements, rather than replaces, the easier-to-interpret one-at-a-time analysis.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_candidate = Path.cwd().resolve()
if repo_candidate.name == 'outbreak_probability_model':
    repo_candidate = repo_candidate.parent
if str(repo_candidate) not in sys.path:
    sys.path.insert(0, str(repo_candidate))

from outbreak_probability_model.london_calibration import (
    DEFAULT_LONDON_SEASONAL_POISSON_FITTED_PARAMETERS, REPO_ROOT,
    SENSITIVITY_PARAMETER_BOUNDS,
    HistoryConditioningConfig,
    forecast_parameter_sensitivity, forecast_parameter_spread_sensitivity,
    load_london_fitted_parameters,
)

output_dir = REPO_ROOT / 'experiments' / 'measles' / 'London' / 'parameter_sensitivity_seasonal_poisson'
output_dir.mkdir(parents=True, exist_ok=True)

# PILOT controls computational cost only. The final setting uses more
# parameter combinations and more SDE repeats, so its variance estimates
# and outbreak probabilities have lower Monte Carlo error.
PILOT = False
RUN_JOINT_GLOBAL_ANALYSIS = True
N_DESIGN = 16 if PILOT else 128
SIMULATIONS_PER_DESIGN = 40 if PILOT else 200
# Separate one-at-a-time experiment: pilot mode uses three levels and 20
# repeats for a quick workflow check; final mode uses seven levels and
# 300 repeats for more stable variance estimates.
# while every other parameter is held at the fitted/selected baseline.
RUN_INDIVIDUAL_PARAMETER_SPREAD = True
LEVELS_PER_PARAMETER = 3 if PILOT else 7
SIMULATIONS_PER_LEVEL = 20 if PILOT else 300
# Each parameter is varied ±20% around the value loaded from the London fit,
# then clipped to its allowed calibration bounds. This is a scenario range,
# not an estimated 95% confidence interval.
RELATIVE_RANGE = 0.20
OUTBREAK_THRESHOLD = 15.0
RANDOM_SEED = 20260824
HISTORY_CONDITIONING = HistoryConditioningConfig(
    history_weeks=4, transmission_multiplier_bounds=(0.8, 1.25),
    regularization_strength=1.0, origin_observation_weight=4.0, maxiter=80,
)
PRIMARY_AGE_GROUP = 'all ages combined'
PRIMARY_OUTCOME = 'peak_weekly_cases'
FITTED_PARAMETER_FILE = DEFAULT_LONDON_SEASONAL_POISSON_FITTED_PARAMETERS

# `fitted` is read from 04_best_fitted_parameters.csv. A value may be a
# genuinely selected fit or a retained starting value; consult the calibration audit.
_, fitted = load_london_fitted_parameters(FITTED_PARAMETER_FILE)
display(pd.DataFrame({
    'parameter': list(fitted),
    'fitted_or_selected_value': list(fitted.values()),
    'global_lower_bound': [SENSITIVITY_PARAMETER_BOUNDS[p][0] for p in fitted],
    'global_upper_bound': [SENSITIVITY_PARAMETER_BOUNDS[p][1] for p in fitted],
}))
print(f'{N_DESIGN} parameter designs x {SIMULATIONS_PER_DESIGN} stochastic paths = {N_DESIGN * SIMULATIONS_PER_DESIGN:,} forecasts')

In [ ]:
# Optional advanced joint-design analysis. The primary OAT analysis below
# is easier to explain and is run by default.
if RUN_JOINT_GLOBAL_ANALYSIS:
    sensitivity = forecast_parameter_sensitivity(
        fitted_parameters_path=FITTED_PARAMETER_FILE,
        relative_range=RELATIVE_RANGE,
        n_design=N_DESIGN,
        simulations_per_design=SIMULATIONS_PER_DESIGN,
        outbreak_threshold=OUTBREAK_THRESHOLD,
        horizon_weeks=6, warmup_weeks=0, random_seed=RANDOM_SEED,
        history_conditioning=HISTORY_CONDITIONING,
    )
    sensitivity.design.to_csv(output_dir / '01_parameter_design.csv', index=False)
    sensitivity.simulation_outcomes.to_csv(output_dir / '02_simulation_outcomes.csv', index=False)
    sensitivity.variance_decomposition.to_csv(output_dir / '03_variance_decomposition.csv', index=False)
    sensitivity.parameter_ranking.to_csv(output_dir / '04_parameter_ranking.csv', index=False)
    sampled_ranges = sensitivity.design.drop(columns='design_id').agg(['min', 'max']).T
    sampled_ranges.index.name = 'parameter'
    display(sampled_ranges)
    sampled_ranges.to_csv(output_dir / '00_actual_sampled_ranges.csv')
else:
    sensitivity = None
    print('Skipped optional joint-design analysis; running the simpler OAT analysis below.')

## Optional advanced check: joint-design variance decomposition

At each parameter design $\theta$, repeated paths give a conditional mean $E[Y|\theta]$ and conditional variance $Var(Y|\theta)$. Across designs,

$$Var(Y) \approx Var_{\theta}(E[Y|\theta]) + E_{\theta}(Var(Y|\theta)).$$

- **Red / parameter changes:** variance of the design-specific mean predictions. It answers: ‘within the tested ±20% ranges, how much does changing parameters move the expected outcome?’
- **Grey / stochastic SDE noise:** average variance among repeated paths using exactly the same parameter values. It includes SDE process noise and weekly count sampling.
- **Total predictive variance:** red variance plus grey variance.

For example, red = 0.40 means 40% of the modelled predictive variance came from changing parameter scenarios and 60% from stochastic paths. It does **not** mean parameters caused 40% of cases. The result applies only to the parameter ranges, initialisation, noise setting and six-week horizon used here; it does not include every real-world uncertainty.

For `outbreak_indicator`, each path is coded 1 if its maximum weekly cases is strictly greater than the threshold and 0 otherwise. With threshold 10, exactly 10 does not count. The decomposition is therefore on a binary event. The other outcomes are case counts.

In [ ]:
if sensitivity is None:
    print('Joint variance decomposition skipped. The simple per-parameter variance is reported below.')
else:
    all_age_variance = sensitivity.variance_decomposition.query("age_group == 'all ages combined'").copy()
    variance_columns = [
        'outcome', 'between_parameter_variance', 'mean_stochastic_variance',
        'total_predictive_variance', 'parameter_variance_fraction',
        'stochastic_variance_fraction',
    ]
    display(all_age_variance[variance_columns])

# Print the graph's meaning in ordinary language using the calculated values.
    for row in all_age_variance.itertuples():
        print(
            f'{row.outcome}: {100 * row.parameter_variance_fraction:.1f}% of modelled variance ' 
            f'came from the tested parameter changes and ' 
            f'{100 * row.stochastic_variance_fraction:.1f}% from repeated stochastic paths.'
        )

    plot_data = all_age_variance.set_index('outcome')[[
        'parameter_variance_fraction', 'stochastic_variance_fraction'
    ]]
    ax = plot_data.plot.barh(stacked=True, figsize=(10, 4), color=['#d62728', '#7f7f7f'])
    ax.set(xlabel='Fraction of total predictive variance', ylabel='Forecast outcome', xlim=(0, 1),
           title='London all ages: predictive variance decomposition')
    ax.legend(['tested parameter scenarios', 'stochastic paths at fixed parameters'],
              loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2)
    for container in ax.containers:
        labels = [f'{100 * value:.1f}%' if value >= 0.04 else '' for value in container.datavalues]
        ax.bar_label(container, labels=labels, label_type='center', color='white', fontsize=9)
    plt.tight_layout(); plt.savefig(output_dir / '05_variance_decomposition.png', dpi=180, bbox_inches='tight'); plt.show()

## Primary analysis: simple individual-parameter variance and outcome spread

The joint design above is useful for screening interactions, but it does not give a clean isolated number for each parameter. This section performs a separate **one-at-a-time (OAT)** experiment:

1. Choose one parameter.
2. Hold every other parameter at its fitted/selected baseline.
3. Test `LEVELS_PER_PARAMETER` values from the lower to upper sensitivity bound, including the exact baseline.
4. Run `SIMULATIONS_PER_LEVEL` stochastic paths at every value.
5. Repeat for every fitted or declared fixed sensitivity parameter.

### The two numbers to present

Let $p_{j,l}$ be the mean simulated outcome when parameter $j$ is set to level $l$. The simplest absolute measure is

$$\text{spread}_j = \max_l(p_{j,l})-\min_l(p_{j,l}).$$

For outbreak probability, multiplying this by 100 gives percentage points. The variance measure is the variance of the level means after subtracting the estimated Monte Carlo variance of those means:

$$V_j=\max\left[\operatorname{Var}_l(p_{j,l})-\overline{\operatorname{Var}(Y|l)}/n,0\right].$$

The single percentage shown in the summary plot is

$$\text{relative share}_j=100\times V_j/\sum_k V_k.$$

These shares sum to 100% across the tested parameters. They answer which parameter explains the largest share of **parameter-driven variation in this OAT experiment**. They do not include interactions and must always be reported together with the tested ranges.

For parameter $i$ and outcome $Y$ the notebook reports:

- **Outcome mean spread:** $\max_l E[Y|\theta_i=l]-\min_l E[Y|\theta_i=l]$. This is the easiest absolute effect size: percentage points for outbreak probability and cases for count outcomes.
- **Raw variance across level means:** variance of the conditional mean outcomes across the tested values.
- **Monte Carlo variance of a level mean:** average within-level stochastic variance divided by paths per level.
- **Noise-adjusted parameter variance:** `max(raw between-level variance − Monte Carlo variance of a level mean, 0)`. This prevents ordinary simulation error from being counted as parameter sensitivity.
- **Mean within-level stochastic variance:** average path variance when that parameter value and all other values are fixed.
- **OAT parameter variance fraction:** noise-adjusted parameter variance divided by itself plus within-level stochastic variance.
- **Isolated variance screening share:** each parameter's noise-adjusted OAT variance divided by the sum across parameters for the same outcome. This sums to 1 for ranking, but OAT effects are not additive when parameters interact, so it is not a Sobol index.

These values apply only to the chosen sensitivity ranges. Doubling a range will generally increase its measured spread, so always report `tested_lower` and `tested_upper` with the result.

In [ ]:
if RUN_INDIVIDUAL_PARAMETER_SPREAD:
    individual_spread = forecast_parameter_spread_sensitivity(
        fitted_parameters_path=FITTED_PARAMETER_FILE,
        relative_range=RELATIVE_RANGE,
        levels_per_parameter=LEVELS_PER_PARAMETER,
        simulations_per_level=SIMULATIONS_PER_LEVEL,
        outbreak_threshold=OUTBREAK_THRESHOLD,
        horizon_weeks=6,
        warmup_weeks=0,
        random_seed=RANDOM_SEED + 1,
        sample_weekly_counts=True,
        history_conditioning=HISTORY_CONDITIONING,
    )
    individual_spread.parameter_levels.to_csv(output_dir / '08_oat_parameter_levels.csv', index=False)
    individual_spread.simulation_outcomes.to_csv(output_dir / '09_oat_simulation_outcomes.csv', index=False)
    individual_spread.level_summary.to_csv(output_dir / '10_oat_level_summary.csv', index=False)
    individual_spread.parameter_spread.to_csv(output_dir / '11_oat_parameter_spread.csv', index=False)

    # One numeric row per parameter and outcome for all-age London.
    oat_all_age = individual_spread.parameter_spread.query(
        "age_group == 'all ages combined' and outcome in ['outbreak_indicator', 'peak_weekly_cases', 'cumulative_cases']"
    ).copy()
    oat_columns = [
        'outcome', 'parameter', 'tested_lower', 'tested_upper',
        'baseline_outcome_mean', 'minimum_level_mean', 'maximum_level_mean',
        'outcome_mean_spread', 'outbreak_probability_spread_percentage_points',
        'noise_adjusted_parameter_variance',
        'mean_within_level_stochastic_variance',
        'oat_parameter_variance_fraction',
        'isolated_variance_screening_share', 'sensitivity_rank',
    ]
    display(oat_all_age[oat_columns].sort_values(['outcome', 'sensitivity_rank']))

    # Print the largest isolated spread for each outcome in its natural units.
    for outcome, rows in oat_all_age.groupby('outcome'):
        top = rows.sort_values('noise_adjusted_parameter_variance', ascending=False).iloc[0]
        if outcome == 'outbreak_indicator':
            spread_text = f'{top.outbreak_probability_spread_percentage_points:.1f} percentage points'
        else:
            spread_text = f'{top.outcome_mean_spread:.2f} cases'
        print(
            f'{outcome}: {top.parameter} had the largest noise-adjusted OAT variance; ' 
            f'the mean outcome spread across its tested range was {spread_text}.'
        )

    # Primary presentation table and plot: one relative variance share per parameter.
    plot_outcome = PRIMARY_OUTCOME
    simple = oat_all_age.query('outcome == @plot_outcome').copy()
    if simple.relative_parameter_variance_share_percent.sum() <= 0:
        plot_outcome = 'peak_weekly_cases'
        simple = oat_all_age.query('outcome == @plot_outcome').copy()
        print('The outbreak probability was constant across all tested parameter levels, so its variance is zero.')
        print('The summary plot therefore uses peak weekly cases, which is not saturated at the threshold.')
    # Omit mu because it duplicates the contact-scale pathway, and omit dt
    # because it is a numerical time-step check rather than epidemiology.
    simple = simple.loc[~simple.parameter.isin({'mu', 'dt'})].copy()
    simple['parameter_vs_stochastic_percent'] = 100 * simple.oat_parameter_variance_fraction
    if plot_outcome == 'outbreak_indicator':
        simple['absolute_effect'] = simple.outbreak_probability_spread_percentage_points
        effect_unit = 'percentage-point probability spread'
    else:
        simple['absolute_effect'] = simple.outcome_mean_spread
        effect_unit = 'case spread'
    variance_total = simple.noise_adjusted_parameter_variance.sum()
    simple['relative_parameter_variance_share_percent'] = np.where(
        variance_total > 0, 100 * simple.noise_adjusted_parameter_variance / variance_total, 0.0
    )
    simple = simple.sort_values('relative_parameter_variance_share_percent', ascending=False)
    simple_columns = [
        'parameter', 'tested_lower', 'tested_upper', 'baseline_outcome_mean',
        'low_level_outcome_mean', 'high_level_outcome_mean', 'absolute_effect',
        'noise_adjusted_parameter_variance',
        'relative_parameter_variance_share_percent', 'parameter_vs_stochastic_percent',
    ]
    display(simple[simple_columns])
    simple[simple_columns].to_csv(output_dir / '12_simple_sensitivity_summary.csv', index=False)

    readable_names = {
        'contact_scale': 'Overall contact-matrix multiplier',
        'sigma': 'Susceptible-state transmission coefficient (σ)',
        'local_mixing': 'Share of transmission from local mixing',
        'seasonal_period_weeks': 'Seasonal cycle length (weeks)',
        'reporting_rate': 'Confirmed-case reporting fraction',
        'gamma': 'Exposed → infectious progression rate (γ, per day)',
        'sick_contact_multiplier': 'Relative contact rate while sick',
        'seasonal_peak_week': 'Week of maximum seasonal transmission',
        'phi': 'Recovery rate from the sick state (φ, per day)',
        'seasonal_amplitude': 'Strength of seasonal transmission forcing',
        'seed_infections_per_week': 'External infections introduced per week',
        'psi': 'Infectious → sick progression rate (ψ, per day)',
        'sick_mobility_multiplier': 'Relative mobility while sick',
        'nu': 'Disease-related mortality rate while sick (ν, per day)',
        'noise_scale': 'Overall stochastic (SDE) noise strength',
        'initial_infectious_per_case': 'Initial infectious people per observed case',
        'initial_exposed_per_case': 'Initial exposed people per observed case',
        'eta': 'Natural mortality rate (η, per day)',
        'sh_noise_multiplier': 'Susceptible/protected-state noise multiplier',
        'delta': 'Protected-state transmission coefficient (δ)',
        'beta_0': 'Birth/recruitment rate (β₀, per day)',
    }
    plot_data = simple.sort_values('relative_parameter_variance_share_percent').copy()
    plot_data['plot_label'] = plot_data.parameter.map(readable_names).fillna(
        plot_data.parameter.str.replace('_', ' ', regex=False)
    )
    fig, ax = plt.subplots(figsize=(13, max(5, 0.42 * len(plot_data))))
    bars = ax.barh(
        plot_data.plot_label, plot_data.relative_parameter_variance_share_percent,
        color='#d62728', alpha=.84,
    )
    labels = [
        f'{share:.1f}%   |   forecast range {effect:.1f} cases'
        for share, effect in zip(
            plot_data.relative_parameter_variance_share_percent, plot_data.absolute_effect
        )
    ]
    ax.bar_label(bars, labels=labels, padding=4, fontsize=8)
    largest = max(float(plot_data.relative_parameter_variance_share_percent.max()), 1.0)
    ax.set_xlim(0, min(125, largest * 1.55 + 5))
    ax.set_xlabel('Relative contribution to parameter-driven variation (%)')
    ax.set_ylabel('')
    ax.set_title(
        'Sensitivity of the six-week peak weekly case forecast\n'
        'Longer bars indicate a larger change across the tested parameter range'
    )
    ax.grid(axis='x', alpha=.2)
    fig.tight_layout()
    fig.savefig(output_dir / 'sensitivity_summary.png', dpi=180, bbox_inches='tight')
    plt.show()
else:
    print('Skipped individual parameter spread; set RUN_INDIVIDUAL_PARAMETER_SPREAD=True to run it.')

## Optional advanced check: joint-design parameter screening

For every parameter, the notebook correlates its design values with the mean outcome at each design using Spearman's rank correlation $\rho$.

- $\rho$ near +1: larger parameter values are strongly associated with larger outcomes.
- $\rho$ near −1: larger parameter values are strongly associated with smaller outcomes.
- $\rho$ near 0: little monotonic relationship over the tested range.

`sensitivity_rank` orders parameters by $|\rho|$. `screening_importance_share` squares each correlation and divides it by the sum across parameters. It is only a convenient relative screening score: 0.40 means 40% of the summed squared marginal correlations, not 40% of forecast variance. Correlated effects, interactions and non-monotonic relationships can make this score misleading; use it to decide what deserves a more rigorous follow-up, not as a causal percentage.

In [ ]:
if sensitivity is None:
    print('Joint correlation screening skipped by default.')
else:
    ranking = sensitivity.parameter_ranking.query(
        "age_group == 'all ages combined' and outcome in ['outbreak_indicator', 'peak_weekly_cases', 'cumulative_cases']"
    ).copy()
    ranking['screening_share_percent'] = 100 * ranking['screening_importance_share']
    display(ranking[[
        'outcome', 'parameter', 'spearman_rho', 'screening_share_percent',
        'sensitivity_rank'
    ]].sort_values(['outcome', 'sensitivity_rank']))

# Print the leading relationship and its direction for every outcome.
    for outcome, outcome_rows in ranking.groupby('outcome'):
        top = outcome_rows.sort_values('sensitivity_rank').iloc[0]
        direction = 'increased' if top.spearman_rho > 0 else 'decreased'
        print(
            f'{outcome}: strongest screened parameter = {top.parameter}; ' 
            f'rho={top.spearman_rho:.3f}. The outcome generally {direction} as this parameter increased.'
        )

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, outcome in zip(axes, ['outbreak_indicator', 'peak_weekly_cases', 'cumulative_cases']):
        data = ranking.query('outcome == @outcome').copy()
        data['absolute_rho'] = data.spearman_rho.abs()
        data = data.sort_values('absolute_rho')
        colours = np.where(data.spearman_rho >= 0, '#d62728', '#1f77b4')
        ax.barh(data.parameter, data.spearman_rho, color=colours, alpha=0.8)
        ax.axvline(0, color='black', linewidth=.8)
        ax.set_title(outcome.replace('_', ' '))
        ax.set_xlabel('Spearman rho: negative lowers outcome; positive raises outcome')
        ax.set_xlim(-1, 1)
    fig.suptitle('Direction and strength of parameter–outcome screening relationships')
    plt.tight_layout(); plt.savefig(output_dir / '06_parameter_ranking.png', dpi=180, bbox_inches='tight'); plt.show()

## 4. Monte Carlo stability: did we run enough repeated paths?

At design $d$, outbreak probability is $\hat p_d = k_d/n$, where $k_d$ paths crossed the threshold and $n$ paths were run. Its estimated Monte Carlo standard error is

$$SE(\hat p_d)=\sqrt{\hat p_d(1-\hat p_d)/n}.$$

The approximate ±1.96 SE interval below measures numerical simulation precision only. It is not a scientific confidence interval for the true future outbreak probability. Increase `SIMULATIONS_PER_DESIGN` when this error is too large; increase `N_DESIGN` when the parameter ranking is unstable.

In [ ]:
if sensitivity is None:
    oat_probability = individual_spread.level_summary.query(
        "age_group == 'all ages combined' and outcome == 'outbreak_indicator'"
    ).copy()
    oat_probability['monte_carlo_se'] = np.sqrt(
        oat_probability.conditional_mean * (1 - oat_probability.conditional_mean)
        / oat_probability.simulations
    )
    stability = oat_probability
else:
    all_age = sensitivity.simulation_outcomes.query("age_group == 'all ages combined'")
    stability = all_age.groupby('design_id').outbreak_indicator.agg(['mean', 'count']).reset_index()
    stability['monte_carlo_se'] = np.sqrt(stability['mean'] * (1 - stability['mean']) / stability['count'])
    stability = sensitivity.design.merge(stability, on='design_id')
stability.to_csv(output_dir / '07_outbreak_probability_stability.csv', index=False)
display(stability.sort_values('monte_carlo_se', ascending=False))
print('Maximum Monte Carlo SE:', stability.monte_carlo_se.max())
print('Approximate worst-case 95% simulation error: ±', 1.96 * stability.monte_carlo_se.max())

## 5. Interpretation and next fitting step

Use the simple results in this order:

1. Check Monte Carlo error. Do not interpret differences smaller than the simulation error.
2. Use `12_simple_sensitivity_summary.png` as the main presentation figure. Bar length is the parameter's share of noise-adjusted OAT parameter variance; the label also gives the absolute outcome spread.
3. Always state the tested lower and upper values. The percentages are conditional on those ranges.
4. If outbreak probability is 0% or 100% at every tested level, it has zero variance and cannot rank parameters. The notebook automatically summarizes peak weekly cases instead and prints a warning.
5. Prioritise parameters with both a large variance share and a scientifically meaningful absolute spread. Then test whether those parameters are identifiable from London data.

Sensitivity and identifiability are different. Sensitivity says changing a value changes the forecast. Identifiability says the observed data can determine that value separately from other parameters. Influential but unidentifiable parameters need external evidence, a prior, or scenario analysis—not an unsupported point estimate. Parameters with negligible influence over scientifically plausible ranges can normally be fixed.

### Output files

- `00_actual_sampled_ranges.csv`: exact min/max used in this run.
- `01_parameter_design.csv`: every joint parameter design.
- `02_simulation_outcomes.csv`: every stochastic path outcome.
- `03_variance_decomposition.csv`: between-parameter and within-parameter variance.
- `04_parameter_ranking.csv`: signed correlations and screening importance shares.
- `07_outbreak_probability_stability.csv`: probabilities and Monte Carlo precision by design.
- `08`–`10`: OAT parameter levels, individual paths and level summaries.
- `11_oat_parameter_spread.csv`: the requested numerical spread/variance for every parameter, age group and outcome.
- `12_simple_sensitivity_summary.csv`: one presentation-ready row per parameter for the selected outcome.
- `12_simple_sensitivity_summary.png`: the main sensitivity plot, showing relative variance share and absolute spread.